In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

pod_meta = pd.read_excel("../data/raw/podlesny2022_metadata.xlsx")

print(f"Podlesny metadata shape: {pod_meta.shape}")
print(f"Columns: {pod_meta.columns.tolist()}")

print("\nSample types:")
print(pod_meta['Sample_Type'].value_counts())

print("\nDisease groups:")
print(pod_meta['Disease_Group'].value_counts())

print("\nDays since FMT range:")
print(
    f"{pod_meta['Days_Since_FMT'].min():.0f} to "
    f"{pod_meta['Days_Since_FMT'].max():.0f}"
)

Podlesny metadata shape: (1416, 20)
Columns: ['Manuscript', 'Study', 'Disease_Group', 'Disease', 'Case_Name', 'Name', 'Donor.Name', 'Sample_Type', 'Days_Since_FMT', 'Case_ABx', 'ABx_days', 'ABx_PostFMT', 'Lavage', 'FMT_route', 'FMT_rounds', 'Colonoscopy_n', 'Nasogastric_n', 'Gastroduodenal_n', 'Enema_n', 'Pill_n']

Sample types:
Sample_Type
Post            826
Donor           297
Pre No-ABx      216
Pre Post-ABx     77
Name: count, dtype: int64

Disease groups:
Disease_Group
IBD     465
MetS    445
ICI     233
rCDI    153
MDR     120
Name: count, dtype: int64

Days since FMT range:
-29 to 535


In [2]:
pod_meta['Case_Name'] = pod_meta['Case_Name'].astype(str)

rng = np.random.default_rng(42)

all_cases = pod_meta['Case_Name'].unique()
n_test = int(len(all_cases) * 0.20)

test_cases = rng.choice(all_cases, size=n_test, replace=False)
train_cases = np.setdiff1d(all_cases, test_cases)

pod_meta['split'] = pod_meta['Case_Name'].apply(
    lambda c: 'test' if c in test_cases else 'train'
)

print(f"Total cases: {len(all_cases)}")
print(f"Train cases: {len(train_cases)} (80%)")
print(f"Test cases: {len(test_cases)} (20%)")

print("\nSplit by disease:")
print(pod_meta.groupby(['Disease_Group', 'split']).size())

# Save
pod_meta.to_csv("../data/processed/podlesny_meta_split.csv", index=False)
pd.Series(train_cases).to_csv("../data/processed/train_cases.csv", index=False)
pd.Series(test_cases).to_csv("../data/processed/test_cases.csv", index=False)

print("\nSplit locked and saved")

Total cases: 263
Train cases: 211 (80%)
Test cases: 52 (20%)

Split by disease:
Disease_Group  split
IBD            test      84
               train    381
ICI            test      68
               train    165
MDR            test      13
               train    107
MetS           test      65
               train    380
rCDI           test      15
               train    138
dtype: int64

Split locked and saved


In [3]:
print("Loading Podlesny abundance")

pod_long = pd.read_excel("../data/raw/podlesny2022_abundance.xlsx")

print("Columns:", pod_long.columns.tolist())
print("Shape:", pod_long.shape)

pod_long.columns = pod_long.columns.str.strip()

pod_wide = pod_long.pivot_table(
    index='Name',
    columns='species',
    values='rel_abund',
    fill_value=0
)

print("Wide shape:", pod_wide.shape)
print("Samples:", pod_wide.shape[0])
print("Species:", pod_wide.shape[1])

pod_wide.to_pickle("../data/processed/podlesny_wide.pkl")

print("Saved as pickle")

Loading Podlesny abundance
Columns: ['Name', 'species', 'rel_abund']
Shape: (118245, 3)
Wide shape: (1419, 860)
Samples: 1419
Species: 860
Saved as pickle


In [4]:
print(pod_wide.isna().sum().sum())

0


In [5]:
print("Loading data")

meta = pd.read_csv("../data/processed/podlesny_meta_split.csv")

pod_wide = pd.read_pickle("../data/processed/podlesny_wide.pkl")

print("Meta:", meta.shape)
print("Abundance:", pod_wide.shape)

common_samples = set(meta['Name']).intersection(pod_wide.index)

meta = meta[meta['Name'].isin(common_samples)].copy()
pod_wide = pod_wide.loc[meta['Name']]

print("After alignment:")
print("Meta:", meta.shape)
print("Abundance:", pod_wide.shape)

meta = meta.set_index('Name')

print("Data merged and aligned")

Loading data
Meta: (1416, 21)
Abundance: (1419, 860)
After alignment:
Meta: (1416, 21)
Abundance: (1416, 860)
Data merged and aligned


In [6]:
assert all(meta.index == pod_wide.index)

In [7]:
print(meta.columns)

Index(['Manuscript', 'Study', 'Disease_Group', 'Disease', 'Case_Name',
       'Donor.Name', 'Sample_Type', 'Days_Since_FMT', 'Case_ABx', 'ABx_days',
       'ABx_PostFMT', 'Lavage', 'FMT_route', 'FMT_rounds', 'Colonoscopy_n',
       'Nasogastric_n', 'Gastroduodenal_n', 'Enema_n', 'Pill_n', 'split'],
      dtype='object')


In [8]:
print(meta['Sample_Type'].value_counts())

Sample_Type
Post            826
Donor           297
Pre No-ABx      216
Pre Post-ABx     77
Name: count, dtype: int64


In [14]:
print("Deriving engraftment labels")

records = []

for case in meta['Case_Name'].dropna().unique():

    case_rows = meta[meta['Case_Name'] == case]

    if case_rows.empty:
        continue

    if case_rows['Donor.Name'].isna().all():
        continue

    donor_name = case_rows['Donor.Name'].iloc[0]
    split = case_rows['split'].iloc[0]
    disease = case_rows['Disease_Group'].iloc[0]

    if donor_name not in meta.index:
        continue

    donor_id = donor_name

    post_rows = case_rows[case_rows['Sample_Type'] == 'Post']

    if post_rows.empty:
        continue

    late = post_rows[post_rows['Days_Since_FMT'] >= 28]
    post_use = late if not late.empty else post_rows

    if post_use.empty:
        continue

    post_use = post_use.sort_values('Days_Since_FMT')

    post_id = post_use.index[0]
    days = post_use['Days_Since_FMT'].iloc[0]

    if donor_id not in pod_wide.index:
        continue

    if post_id not in pod_wide.index:
        continue

    donor_prof = pod_wide.loc[donor_id]
    if isinstance(donor_prof, pd.DataFrame):
        donor_prof = donor_prof.iloc[0]

    post_prof = pod_wide.loc[post_id]
    if isinstance(post_prof, pd.DataFrame):
        post_prof = post_prof.iloc[0]

    for species in pod_wide.columns:

        val = donor_prof[species]

        if pd.notna(val) and float(val) > 0.001:

            post_val = post_prof[species]

            records.append({
                'case': case,
                'donor_id': donor_id,
                'post_id': post_id,
                'species': species,
                'donor_abundance': float(val),
                'post_abundance': float(post_val),
                'engrafted': int(float(post_val) > 0.001),
                'disease': disease,
                'days_post_fmt': days,
                'split': split
            })

labels_df = pd.DataFrame(records)

labels_df.to_csv("../data/processed/engraftment_labels.csv", index=False)

print(f"Total records: {len(labels_df)}")
print(f"Engraftment rate: {labels_df['engrafted'].mean():.2%}")
print(f"Cases covered: {labels_df['case'].nunique()}")
print(f"Train records: {(labels_df['split']=='train').sum()}")
print(f"Test records: {(labels_df['split']=='test').sum()}")

print("\nBy disease:")
print(labels_df.groupby('disease')['engrafted'].agg(['mean', 'count']).round(3))

print("\nEngraftment labels saved")

Deriving engraftment labels
Total records: 10951
Engraftment rate: 52.33%
Cases covered: 203
Train records: 8722
Test records: 2229

By disease:
          mean  count
disease              
IBD      0.605   1483
ICI      0.596    969
MDR      0.501   1326
MetS     0.489   5606
rCDI     0.543   1567

Engraftment labels saved


In [15]:
print(labels_df.isna().sum().sum())
print(labels_df['engrafted'].value_counts())
print(labels_df['species'].nunique())

0
engrafted
1    5731
0    5220
Name: count, dtype: int64
272
